# Interferometric Coherence and Closure Phase per ISMN Station

<br>

For each ISMN station, reads the per-acquisition GSLC window chips saved by
[`01b_get_nisar_gslc_windows.ipynb`](01b_get_nisar_gslc_windows.ipynb) and
computes, over a small window centered on the station's pixel:

- **pairwise coherence** for every pair of acquisitions from the same track
- **closure phase** for every triplet 

One CSV per station for each, under `nisar_data/<AOI>_GSLC_coherence_closure/`.
Purely local, in-memory -- no network access.

:::{note} Method
Implemented by `process_station()` in
[`util/nisar_coherence_closure.py`](../util/nisar_coherence_closure.py) --
see that module's docstrings for the coherence/closure-phase formulas and
the full column list. No triplet is dropped for low coherence; each carries
`min_coherence_HH`/`min_coherence_HV` so low-coherence triplets can be
filtered downstream without recomputing anything. Both CSVs are fully
rebuilt from scratch on every run.
:::

<hr>

## Overview

1. Load station list
2. Process each station: group by track, compute pairwise coherence and
   triplet closure phase, save CSVs
3. (Optional) Preview one station's output

<hr>

In [1]:
import sys
import os
import json
from pathlib import Path

from util.nisar_coherence_closure import process_station

C:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Load station list

Only the names are needed here -- the coherence/closure-phase computation
reads everything else straight out of each station's saved `.nc` window
files.

In [2]:
# stations.geojson's AOI subfolder -- this example run uses Alaska.
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

station_names = [feat["properties"]["station_val"] for feat in stations_geojson["features"]]

print(f"Loaded {len(station_names)} station names")
station_names

Loaded 20 station names


['MooseInc',
 'Aniak',
 'AtigunPass',
 'Coldfoot',
 'EagleSummit',
 'GobblersKnob',
 'GraniteCrk',
 'ImnaviatCreek',
 'LittleChenaRidge',
 'McGrath',
 'Nenana',
 'NukaGlacier',
 'PilgrimHotSprings',
 'PrudhoeBay',
 'RockyPoint',
 'SpringCreek',
 'SusitnaValleyHigh',
 'TurnagainPass',
 'UpperNomeCreek',
 'Kenai-29-ENE']

## 2. Process each station: group by track, compute coherence and closure phase, save CSVs

`process_station()` groups each station's saved windows by track, computes
HH/HV coherence and closure phase as described in the module note above,
and saves both CSVs under `output_dir`. A station with no saved window
files gets two empty (header-only) CSVs rather than an error.

In [3]:
COHERENCE_WINDOW_HALF = 1  # window half-size for coherence/closure phase estimation; 1 = 3x3 (default), 2 = 5x5, etc.
# Larger reduces the small-sample bias in the coherence estimate (worst at low true coherence with the 3x3 default)
# at the cost of spatial resolution around the station.

# Reads the real saved windows for this example run's AOI (Alaska).
gslc_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"
output_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_coherence_closure"

for station_name in station_names:
    coherence_path, closure_path = process_station(
        station_name, gslc_dir, output_dir, window_half=COHERENCE_WINDOW_HALF
    )
    n_coherence = sum(1 for _ in open(coherence_path)) - 1
    n_closure = sum(1 for _ in open(closure_path)) - 1
    print(f"{station_name}: {n_coherence} pair(s) -> {coherence_path.name}, "
          f"{n_closure} triplet(s) -> {closure_path.name}")

MooseInc: 36 pair(s) -> MooseInc_NISAR_GSLC_coherence.csv, 34 triplet(s) -> MooseInc_NISAR_GSLC_closure_phase.csv
Aniak: 0 pair(s) -> Aniak_NISAR_GSLC_coherence.csv, 0 triplet(s) -> Aniak_NISAR_GSLC_closure_phase.csv


AtigunPass: 32 pair(s) -> AtigunPass_NISAR_GSLC_coherence.csv, 28 triplet(s) -> AtigunPass_NISAR_GSLC_closure_phase.csv


Coldfoot: 36 pair(s) -> Coldfoot_NISAR_GSLC_coherence.csv, 34 triplet(s) -> Coldfoot_NISAR_GSLC_closure_phase.csv


EagleSummit: 52 pair(s) -> EagleSummit_NISAR_GSLC_coherence.csv, 48 triplet(s) -> EagleSummit_NISAR_GSLC_closure_phase.csv


GobblersKnob: 36 pair(s) -> GobblersKnob_NISAR_GSLC_coherence.csv, 34 triplet(s) -> GobblersKnob_NISAR_GSLC_closure_phase.csv


...

SpringCreek: 34 pair(s) -> SpringCreek_NISAR_GSLC_coherence.csv, 26 triplet(s) -> SpringCreek_NISAR_GSLC_closure_phase.csv


SusitnaValleyHigh: 40 pair(s) -> SusitnaValleyHigh_NISAR_GSLC_coherence.csv, 30 triplet(s) -> SusitnaValleyHigh_NISAR_GSLC_closure_phase.csv


TurnagainPass: 38 pair(s) -> TurnagainPass_NISAR_GSLC_coherence.csv, 32 triplet(s) -> TurnagainPass_NISAR_GSLC_closure_phase.csv


UpperNomeCreek: 57 pair(s) -> UpperNomeCreek_NISAR_GSLC_coherence.csv, 58 triplet(s) -> UpperNomeCreek_NISAR_GSLC_closure_phase.csv


Kenai-29-ENE: 32 pair(s) -> Kenai-29-ENE_NISAR_GSLC_coherence.csv, 28 triplet(s) -> Kenai-29-ENE_NISAR_GSLC_closure_phase.csv


## 3. (Optional) Preview one station's output

In [4]:
import pandas as pd

preview_station = station_names[0]
coherence_df = pd.read_csv(output_dir / f"{preview_station}_NISAR_GSLC_coherence.csv")
closure_df = pd.read_csv(output_dir / f"{preview_station}_NISAR_GSLC_closure_phase.csv")

print(f"--- {preview_station} coherence (pairs) ---")
display(coherence_df)
print(f"\n--- {preview_station} closure phase (triplets) ---")
display(closure_df)

--- MooseInc coherence (pairs) ---


,date_1,date_2,temporal_baseline_days,orbit_dir,orbit_num,local_incidence_angle_deg,coherence_HH,phase_HH,coherence_HV,phase_HV
0,2026-06-27T05:52:57,2026-07-09T05:52:56,11.999988,Descending,15,45.806715,0.530313,0.740812,0.401262,2.174185
1,2026-06-27T05:52:57,2026-07-21T05:52:56,23.999988,Descending,15,45.806953,0.645623,-1.858867,0.798244,-2.206730
2,2026-06-27T05:52:57,2026-08-14T05:52:54,47.999965,Descending,15,45.809954,0.116891,-2.543220,0.269718,0.169968
3,2026-07-09T05:52:56,2026-07-21T05:52:56,12.000000,Descending,15,45.804586,0.560371,-1.779822,0.660659,1.997276
4,2026-07-09T05:52:56,2026-08-14T05:52:54,35.999977,Descending,15,45.807587,0.352010,-2.394195,0.402953,-2.226812
5,2026-07-21T05:52:56,2026-08-14T05:52:54,23.999977,Descending,15,45.807825,0.284550,-1.291278,0.133732,1.430785
6,2026-06-19T14:30:45,2026-07-01T14:30:44,11.999988,Ascending,78,37.931900,0.681133,0.327417,0.694660,-2.135794
7,2026-06-19T14:30:45,2026-07-13T14:30:44,23.999988,Ascending,78,37.931536,0.453750,1.892721,0.140997,1.002024
8,2026-06-19T14:30:45,2026-07-25T14:30:43,35.999977,Ascending,78,37.930729,0.171612,0.249114,0.150346,2.528446
9,2026-06-19T14:30:45,2026-08-18T14:30:42,59.999965,Ascending,78,37.928169,0.060724,-2.484441,0.212532,0.515429



--- MooseInc closure phase (triplets) ---


,date_1,date_2,date_3,temporal_baseline_12_days,temporal_baseline_23_days,temporal_baseline_13_days,orbit_dir,orbit_num,local_incidence_angle_deg,closure_phase_HH,closure_phase_HV,min_coherence_HH,min_coherence_HV
0,2026-06-27T05:52:57,2026-07-09T05:52:56,2026-07-21T05:52:56,11.999988,12.000000,23.999988,Descending,15,45.806085,0.819857,0.095006,0.530313,0.401262
1,2026-06-27T05:52:57,2026-07-09T05:52:56,2026-08-14T05:52:54,11.999988,35.999977,47.999965,Descending,15,45.808085,0.889837,-0.222595,0.116891,0.269718
2,2026-06-27T05:52:57,2026-07-21T05:52:56,2026-08-14T05:52:54,23.999988,23.999977,47.999965,Descending,15,45.808244,-0.606925,-0.945914,0.116891,0.133732
3,2026-07-09T05:52:56,2026-07-21T05:52:56,2026-08-14T05:52:54,12.000000,23.999977,35.999977,Descending,15,45.806666,-0.676905,-0.628313,0.284550,0.133732
4,2026-06-19T14:30:45,2026-07-01T14:30:44,2026-07-13T14:30:44,11.999988,12.000000,23.999988,Ascending,78,37.932608,0.866670,0.706333,0.324555,0.140997
5,2026-06-19T14:30:45,2026-07-01T14:30:44,2026-07-25T14:30:43,11.999988,23.999988,35.999977,Ascending,78,37.932070,-1.584383,1.749648,0.142823,0.150346
6,2026-06-19T14:30:45,2026-07-01T14:30:44,2026-08-18T14:30:42,11.999988,47.999977,59.999965,Ascending,78,37.930364,2.707212,-1.516374,0.060724,0.147107
7,2026-06-19T14:30:45,2026-07-13T14:30:44,2026-07-25T14:30:43,23.999988,11.999988,35.999977,Ascending,78,37.931828,1.635644,-0.512645,0.171612,0.140997
8,2026-06-19T14:30:45,2026-07-13T14:30:44,2026-08-18T14:30:42,23.999988,35.999977,59.999965,Ascending,78,37.930121,1.968403,-0.595683,0.060724,0.140997
9,2026-06-19T14:30:45,2026-07-25T14:30:43,2026-08-18T14:30:42,35.999977,23.999988,59.999965,Ascending,78,37.929583,0.584878,1.518847,0.060724,0.150346


<hr>

## Summary

You now have, under `nisar_data/<AOI>_GSLC_coherence_closure/`, two CSVs
per station: `{station}_NISAR_GSLC_coherence.csv` (every pair,
`coherence_HH`/`coherence_HV`) and `{station}_NISAR_GSLC_closure_phase.csv`
(every triplet, `closure_phase_HH`/`_HV` + `min_coherence_HH`/`_HV`) -- see
the module note above for the full column list.

Re-running after more GSLC windows are downloaded simply rebuilds both
CSVs from whatever `.nc` files currently exist -- safe to re-run any time.

<hr>